In [1]:
from PIL import Image, UnidentifiedImageError
import requests
from io import BytesIO
import os
import re
import time

img_url = "https://www.gannett-cdn.com/-mm-/921fd0867919f288546f951b38fb4f16f76f0b69/c=415-0-3429-2266/local/-/media/2017/01/18/USATODAY/USATODAY/636203373357431686-AFP-AFP-K41BC.jpg"

def load_image_from_url(url, verbose=False):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, timeout=10, headers=headers)
        if verbose:
            response.raise_for_status()  # Raises HTTPError for bad status codes
        
        # content_type = response.headers.get("Content-Type", "")
        # if "image" not in content_type:
        #     print(f"URL does not point to an image: {url}")
        #     return None
        
        return Image.open(BytesIO(response.content))
    
    except UnidentifiedImageError:
        if verbose:
            print(f"Cannot identify image from URL: {url}")
        return None
    except Exception as e:
        if verbose:
            print(f"Error loading image from {url}: {e}")
        return None
    
    except Exception as e:
        if verbose:
            print(f"Error loading image from {url}: {e}")
        return None

# load_image_from_url(img_url, verbose=True)

In [2]:
base_path = "nbs/other/images"
# img_url = "https://factuel.afp.com/sites/default/files/styles/twitter_card/public/medias/factchecking/g2/2022-04/2d60c1d679dc1bc79648fdcff6adfaef.jpeg?itok=DLDzvGGQ"

def process_image_url(img_url, base_path):
    # url_until_ext = re.search(r"\.(gif|jpe?g|tiff?|png|webp|bmp)", img_url)
    path_url = img_url.split("/")
    path_url_base = path_url[2:-1]
    filename = path_url[-1]
    filename_until_ext = re.search(r"\.(gif|jpe?g|tiff?|png|webp|bmp)", filename)
    if filename_until_ext:
        filename_until_ext = filename[:filename_until_ext.end()]
    else:
        filename_until_ext = filename + ".jpg"
        
    filename_short = filename_until_ext[-50:] if len(filename_until_ext) > 50 else filename_until_ext
    url_until_ext = "/".join(path_url_base) + "/" + filename_short
    path = f"{base_path}/{url_until_ext}"
    return path

path = process_image_url(img_url, base_path)
path

'nbs/other/images/leadstories.com/caption_3479196.jpg'

In [8]:

def load_save_image(img_url, base_path="nbs/other/images", verbose=False, wait=0):
    time.sleep(wait)
    # url_until_ext = re.search(r"\.(gif|jpe?g|tiff?|png|webp|bmp)", img_url)
    path = process_image_url(img_url, base_path)
    
    os.makedirs(os.path.dirname(path), exist_ok=True)
    img = load_image_from_url(img_url)

    # try:
    if img:
        img.save(path)
        return path
    else:
        if verbose:
            print("Could not load image:", img_url)
        return None

load_save_image(img_url)


'nbs/other/images/leadstories.com/caption_3479196.jpg'

In [6]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

df = pd.read_csv("nbs/other/claim_w_verified_images.csv")  # replace with your path
df = df.fillna("")

# df_verified_w_images = pd.read_csv("nbs/other/verified_images.csv")  # replace with your path
# df = df.merge(df_verified_w_images[["url", "image_path"]], on="url", how="left")
# df.to_csv("nbs/other/claim_w_verified_images.csv", index=False)  # replace with your path
# df_verified = df[["url", "meta_image"]].drop_duplicates(subset=["url"])
# df_verified["image_path"] = df_verified["meta_image"].progress_apply(lambda x: load_save_image(x, base_path=base_path, wait=0.5))